# Gestión de errores - Finalize Pipeline Control

## Objetivo

Este notebook tiene como objetivo registrar y actualizar el detalle de errores
generados durante la ejecución del Task `finalize_pipeline_control`.

El proceso permite mantener la trazabilidad de la ejecución en:

`prod_dataco.metadata.etl_log`

## Flujo

1. Recibe `log_id` y `run_id` de la ejecución fallida.
2. Si `log_id` no es recibido, busca el último registro en estado `EN_PROCESO`
   correspondiente a `finalize_pipeline_control`.
3. Utiliza `run_id` para consultar la API de Databricks Jobs.
4. Obtiene el detalle técnico del error generado durante la ejecución.
5. Construye el mensaje de error con la información de trazabilidad.
6. Actualiza el registro correspondiente en `etl_log`.
7. Cambia el estado de `EN_PROCESO` a `ERROR`.

## Tabla de trazabilidad

- Tabla: `prod_dataco.metadata.etl_log`
- Task origen: `finalize_pipeline_control`
- Task de error: `log_error_finalize_control`
- Estado inicial: `EN_PROCESO`
- Estado final en caso de falla: `ERROR`

## Parámetros recibidos

| Parámetro | Descripción |
|---|---|
| `log_id` | Identificador único del registro de trazabilidad |
| `run_id` | Identificador de la ejecución del Task en Databricks |

## Resultado esperado

Ante una falla de `finalize_pipeline_control`, este notebook permite
identificar la ejecución afectada y registrar en `etl_log` el mensaje técnico
devuelto por Databricks.

In [0]:
# ============================================================
# OBTENCIÓN DEL LOG_ID Y PARÁMETROS DEL TASK FALLIDO
# ============================================================

dbutils.widgets.text("error_code", "UNKNOWN_ERROR")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("result_state", "FAILED")

error_code = dbutils.widgets.get("error_code")
result_state = dbutils.widgets.get("result_state")

# Prioriza el run_id numérico publicado por taskValues (si lo publicaste ahí);
# si no, cae al que envía el orquestador como parámetro del Task
try:
    run_id = dbutils.jobs.taskValues.get(
        taskKey="finalize_pipeline_control",
        key="run_id",
        debugValue=""
    )
except Exception:
    run_id = None

if not run_id or run_id == "":
    run_id = dbutils.widgets.get("run_id")

log_id = dbutils.jobs.taskValues.get(
    taskKey="finalize_pipeline_control",
    key="log_id",
    debugValue=""
)

print(f"✓ log_id: {log_id}")
print(f"✓ error_code: {error_code}")
print(f"✓ run_id: {run_id}")
print(f"✓ result_state: {result_state}")

In [0]:
# ============================================================
# RECUPERACIÓN DE LOG_ID EN CASO DE NO HABER SIDO RECIBIDO
# ============================================================

if not log_id:

    print("⚠ log_id no recibido. Buscando ejecución EN_PROCESO...")

    resultado = spark.sql("""
        SELECT log_id
        FROM prod_dataco.metadata.etl_log
        WHERE notebook = 'finalize_pipeline_control'
          AND estado = 'EN_PROCESO'
        ORDER BY fecha_inicio DESC
        LIMIT 1
    """).collect()

    if resultado:
        log_id = resultado[0]["log_id"]
        print(f"✓ log_id recuperado: {log_id}")

    else:
        raise ValueError(
            "No se encontró ningún registro EN_PROCESO "
            "para finalize_pipeline_control."
        )

In [0]:
# ============================================================
# OBTENCIÓN DEL DETALLE DEL ERROR MEDIANTE JOBS API
# ============================================================

import requests

error_detail = None

if run_id and str(run_id).isdigit() and str(run_id) != "0":
    try:
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

        token = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .apiToken()
            .get()
        )

        url = f"https://{workspace_url}/api/2.2/jobs/runs/get-output"

        response = requests.get(
            url,
            headers={"Authorization": f"Bearer {token}"},
            params={"run_id": str(run_id)}
        )

        if response.status_code == 200:
            output_info = response.json()
            error_detail = (
                output_info.get("error")
                or output_info.get("error_trace")
                or (output_info.get("notebook_output") or {}).get("result")
            )
        else:
            error_detail = f"Error al consultar la API (Status {response.status_code}): {response.text}"

    except Exception as ex:
        error_detail = f"Excepción durante la conexión a Jobs API: {str(ex)}"

if not error_detail:
    error_detail = "No fue posible obtener la traza del error mediante Jobs API (Run ID no válido o sin respuesta)."

print("✓ Detalle del error obtenido:")
print(error_detail)

In [0]:
# ============================================================
# CONSTRUCCIÓN DEL MENSAJE DE ERROR
# ============================================================

mensaje_error = (
    f"Task: finalize_pipeline_control | "
    f"Error code: {error_code} | "
    f"Run ID: {run_id} | "
    f"Resultado: {result_state} | "
    f"Detalle: {error_detail}"
)

print("Mensaje de error:")
print(mensaje_error)

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG COMO ERROR
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

delta_log = DeltaTable.forName(
    spark,
    "prod_dataco.metadata.etl_log"
)

if log_id:
    delta_log.update(
        condition=F.col("log_id") == log_id,
        set={
            "fecha_fin": F.current_timestamp(),
            "estado": F.lit("ERROR"),
            "mensaje_error": F.lit(mensaje_error[:2000])
        }
    )
    print(f"✓ etl_log actualizado como ERROR: {log_id}")
else:
    print("⚠️ No se realizó la actualización: log_id no fue recibido.")